# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Choose an appropriate Naive Bayes variant for numeric, count, binary, or categorical features. 
- Configure smoothing, priors, and sample weights in Naive Bayes models. 
- Use partial_fit for incremental Naive Bayes training. 


## **1. Naive Bayes Assumptions**

### **1.1. Conditional Independence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_01.jpg?v=1783967936" width="250">



>* Features act as separate class evidence
>* Simplifies learning by ignoring feature interactions

>* Treat related features separately after class choice
>* Gain stable estimates from simpler patterns

>* Match Naive Bayes variants to feature types
>* Avoid duplicate features that overcount evidence



### **1.2. Gaussian Feature Modeling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_02.jpg?v=1783967941" width="250">



>* Models continuous features with bell-shaped distributions
>* Compares class-specific averages and spreads

>* Gaussian shape is a useful approximation
>* Class averages and spreads guide classification

>* Use Gaussian for continuous measured features
>* Choose other variants for counts or categories



In [ ]:
#@title Python Code - Gaussian Feature Modeling

# Gaussian Naive Bayes models continuous measurements.
# Each class gets feature means and variances.
# New points favor the most plausible class.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny continuous flower measurements.
rng = np.random.default_rng(7)

# Store class names for readable output.
classes = np.array(["short", "tall"])

# Generate heights in centimeters for two classes.
short_heights = rng.normal(150, 6, 24)
tall_heights = rng.normal(180, 7, 24)

# Generate weights in pounds for two classes.
short_weights = rng.normal(120, 10, 24)
tall_weights = rng.normal(170, 12, 24)

# Combine numeric features into one matrix.
X_short = np.column_stack([short_heights, short_weights])
X_tall = np.column_stack([tall_heights, tall_weights])
X = np.vstack([X_short, X_tall])

# Create matching class labels.
y = np.array([0] * 24 + [1] * 24)

# Validate the small teaching dataset.
assert X.shape == (48, 2)
assert y.shape == (48,)

# Estimate Gaussian parameters per class.
means = np.vstack([X[y == k].mean(axis=0) for k in [0, 1]])
variances = np.vstack([X[y == k].var(axis=0) for k in [0, 1]])

# Add tiny smoothing for numerical safety.
variances = variances + 1e-6
priors = np.array([np.mean(y == k) for k in [0, 1]])

# Define one new continuous measurement.
new_person = np.array([[166.0, 145.0]])

# Compute Gaussian log likelihoods feature by feature.
log_terms = -0.5 * np.log(2 * np.pi * variances)
log_terms -= ((new_person - means) ** 2) / (2 * variances)

# Combine feature evidence with class priors.
log_scores = log_terms.sum(axis=1) + np.log(priors)
predicted_index = int(np.argmax(log_scores))

# Print concise teaching results.
print("Gaussian NB uses continuous numeric features.")
print("Features: height centimeters, weight pounds.")
print("Class means:", np.round(means, 1).tolist())
print("New point:", new_person.ravel().tolist())
print("Predicted class:", classes[predicted_index])

# Plot the two Gaussian-modeled feature clouds.
plt.figure(figsize=(6, 4))
plt.scatter(X_short[:, 0], X_short[:, 1], label="short class")
plt.scatter(X_tall[:, 0], X_tall[:, 1], label="tall class")
plt.scatter(new_person[:, 0], new_person[:, 1], s=120, marker="X", label="new point")
plt.xlabel("Height in centimeters")
plt.ylabel("Weight in pounds")
plt.title("Gaussian feature modeling for numeric measurements")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Feature Likelihoods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_01_03.jpg?v=1783967938" width="250">



>* Likelihoods link features to possible classes
>* Match likelihood models to feature measurements

>* Match Naive Bayes variant to feature type
>* Use likelihoods that reflect data meaning

>* Match likelihoods to feature measurement types
>* Handle mixed data with suitable preprocessing



## **2. Naive Bayes Variants**

### **2.1. Multinomial Smoothing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_01.jpg?v=1783967932" width="250">



>* Count features can be missing in training
>* Smoothing prevents zero-probability brittle predictions

>* Pseudo-counts keep feature probabilities balanced
>* Smoothing limits rare-term overconfidence

>* Balance smoothing to avoid overconfidence or underfitting
>* Tune for data size, imbalance, and change



In [ ]:
#@title Python Code - Multinomial Smoothing

# Demonstrate smoothing for multinomial count features.
# Use tiny word counts for two classes.
# Compare zero, weak, and strong smoothing.

import math
import numpy as np

# Store vocabulary and class count totals.
words = ["team", "match", "refund", "invoice"]
classes = ["sports", "billing"]
counts = np.array([[8, 5, 0, 0], [0, 1, 6, 7]], dtype=float)

# Validate the tiny count table shape.
expected_shape = (len(classes), len(words))
assert counts.shape == expected_shape

# Define a safe multinomial probability calculator.
def smoothed_probabilities(count_table, alpha_value):
    feature_count = count_table.shape[1]
    numerator = count_table + alpha_value
    denominator = numerator.sum(axis=1, keepdims=True)

    return numerator / denominator

# Score one short document using log probabilities.
def score_document(prob_table, document_counts):
    safe_logs = np.log(prob_table)
    return safe_logs @ document_counts

# Create a document containing an unseen sports word.
document = np.array([1, 0, 1, 0], dtype=float)
alphas = [0.0, 0.1, 1.0]

# Print compact results for each smoothing value.
print("alpha | P(refund|sports) | predicted class")
for alpha in alphas:
    probs = smoothed_probabilities(counts, alpha)
    scores = score_document(probs, document)
    predicted = classes[int(np.argmax(scores))]

    refund_index = words.index("refund")
    refund_prob = probs[0, refund_index]
    print(f"{alpha:4.1f} | {refund_prob:16.4f} | {predicted}")

# Show how smoothing prevents impossible zero probabilities.
print("Smoothing adds pseudo-counts to every class-feature pair.")



### **2.2. Complement NB**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_02.jpg?v=1783967930" width="250">



>* Improves Naive Bayes for imbalanced classes
>* Learns from other classes for stability

>* Smoothing limits rare feature overconfidence
>* Tune smoothing to protect minority classes

>* Set priors to reflect real class rates
>* Weight samples by reliability, recency, or cost



In [ ]:
#@title Python Code - Complement NB

# Complement NB handles imbalanced count features.
# Smoothing prevents extreme zero count effects.
# Priors and weights change class influence.

import math
from collections import defaultdict

# Store tiny count based training examples.
texts = ["refund refund urgent", "refund billing", "billing invoice", "invoice payment", "crash error", "error bug"]
labels = ["complaint", "normal", "normal", "normal", "technical", "technical"]
weights = [3.0, 1.0, 1.0, 1.0, 1.5, 1.5]

# Build a small vocabulary deterministically.
vocab = sorted({word for text in texts for word in text.split()})
classes = sorted(set(labels))
assert len(vocab) > 0 and len(classes) == 3

# Convert each message into word counts.
def vectorize(text, vocab):
    counts = dict.fromkeys(vocab, 0.0)
    for word in text.split():
        if word in counts:
            counts[word] += 1.0
    return counts

# Train Complement NB with smoothing and priors.
def train_complement_nb(texts, labels, weights, alpha, priors):
    class_counts = {c: defaultdict(float) for c in classes}
    class_totals = dict.fromkeys(classes, 0.0)
    for text, label, weight in zip(texts, labels, weights):
        counts = vectorize(text, vocab)
        for word, value in counts.items():
            class_counts[label][word] += weight * value
            class_totals[label] += weight * value

    total_counts = sum(class_totals.values())
    scores = {c: {} for c in classes}
    for c in classes:
        complement_total = total_counts - class_totals[c]
        denom = complement_total + alpha * len(vocab)
        for word in vocab:
            complement_count = sum(class_counts[o][word] for o in classes if o != c)
            scores[c][word] = math.log((complement_count + alpha) / denom)
    return scores, priors

# Predict by penalizing complement evidence.
def predict(text, model):
    scores, priors = model
    counts = vectorize(text, vocab)
    results = {}
    for c in classes:
        value = math.log(priors[c])
        value -= sum(counts[w] * scores[c][w] for w in vocab)
        results[c] = value
    return max(results, key=results.get), results

# Compare two smoothing and prior settings.
model_a = train_complement_nb(texts, labels, weights, 1.0, {"complaint": 0.10, "normal": 0.70, "technical": 0.20})
model_b = train_complement_nb(texts, labels, weights, 0.2, {"complaint": 0.30, "normal": 0.50, "technical": 0.20})
message = "refund urgent billing"

# Print concise teaching results.
pred_a, scores_a = predict(message, model_a)
pred_b, scores_b = predict(message, model_b)
print("Vocabulary:", vocab)
print("Message:", message)
print("Alpha 1.0, deployment priors ->", pred_a)
print("Alpha 0.2, higher complaint prior ->", pred_b)
print("Sample weights used:", weights)
print("Complement NB uses other classes to score each class.")



### **2.3. Bernoulli Categorical**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_02_03.jpg?v=1783967934" width="250">



>* Model yes-or-no feature presence
>* Use smoothing for stable probabilities

>* Model fixed category values per class
>* Tune smoothing to handle unseen categories

>* Priors set starting class expectations
>* Sample weights emphasize important training examples



In [ ]:
#@title Python Code - Bernoulli Categorical

# Compare Bernoulli and categorical Naive Bayes.
# Smoothing prevents impossible unseen feature values.
# Priors and weights change class influence.

import math
from collections import defaultdict

# Store tiny binary symptom training examples.
bernoulli_X = [[1, 0], [1, 1], [0, 1], [0, 0]]
bernoulli_y = ["spam", "spam", "ham", "ham"]
bernoulli_weights = [2.0, 1.0, 1.0, 1.0]

# Store tiny categorical customer training examples.
categorical_X = [["mobile", "west"], ["desktop", "west"], ["tablet", "east"], ["mobile", "east"]]
categorical_y = ["buy", "buy", "skip", "skip"]
categorical_weights = [1.0, 2.0, 1.0, 1.0]

# Define shared class prior beliefs.
class_priors = {"spam": 0.40, "ham": 0.60, "buy": 0.50, "skip": 0.50}
alpha = 1.0

# Validate matching training lengths.
assert len(bernoulli_X) == len(bernoulli_y) == len(bernoulli_weights)
assert len(categorical_X) == len(categorical_y) == len(categorical_weights)

# Fit Bernoulli probabilities with smoothing.
def fit_bernoulli(X, y, weights, alpha):
    classes = sorted(set(y))
    feature_count = len(X[0])
    totals = defaultdict(float)
    ones = {c: [0.0] * feature_count for c in classes}

    for row, label, weight in zip(X, y, weights):
        totals[label] += weight
        for index, value in enumerate(row):
            ones[label][index] += weight * value

    probs = {}
    for label in classes:
        denom = totals[label] + 2 * alpha
        probs[label] = [(count + alpha) / denom for count in ones[label]]
    return classes, probs

# Fit categorical probabilities with smoothing.
def fit_categorical(X, y, weights, alpha):
    classes = sorted(set(y))
    values = [sorted(set(row[i] for row in X)) for i in range(len(X[0]))]
    totals = defaultdict(float)
    counts = {c: [defaultdict(float) for _ in values] for c in classes}

    for row, label, weight in zip(X, y, weights):
        totals[label] += weight
        for index, value in enumerate(row):
            counts[label][index][value] += weight

    probs = {c: [] for c in classes}
    for label in classes:
        for index, choices in enumerate(values):
            denom = totals[label] + alpha * len(choices)
            probs[label].append({v: (counts[label][index][v] + alpha) / denom for v in choices})
    return classes, values, probs

# Predict using Bernoulli likelihoods and priors.
def predict_bernoulli(row, classes, probs, priors):
    scores = {}
    for label in classes:
        score = math.log(priors[label])
        for value, prob in zip(row, probs[label]):
            score += math.log(prob if value else 1 - prob)
        scores[label] = score
    return max(scores, key=scores.get)

# Predict using categorical likelihoods and priors.
def predict_categorical(row, classes, probs, priors):
    scores = {}
    for label in classes:
        score = math.log(priors[label])
        for index, value in enumerate(row):
            score += math.log(probs[label][index].get(value, 1e-9))
        scores[label] = score
    return max(scores, key=scores.get)

# Train both small teaching models.
b_classes, b_probs = fit_bernoulli(bernoulli_X, bernoulli_y, bernoulli_weights, alpha)
c_classes, c_values, c_probs = fit_categorical(categorical_X, categorical_y, categorical_weights, alpha)

# Make two simple predictions.
bernoulli_prediction = predict_bernoulli([1, 0], b_classes, b_probs, class_priors)
categorical_prediction = predict_categorical(["desktop", "east"], c_classes, c_probs, class_priors)

# Print concise teaching results.
print("Bernoulli classes:", b_classes)
print("P(feature present | spam):", [round(v, 2) for v in b_probs["spam"]])
print("Bernoulli prediction for [present, absent]:", bernoulli_prediction)
print("Categorical classes:", c_classes)
print("P(device=desktop | buy):", round(c_probs["buy"][0]["desktop"], 2))
print("Categorical prediction for desktop/east:", categorical_prediction)



## **3. Incremental NB Training**

### **3.1. Smoothing for Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_01.jpg?v=1783967946" width="250">



>* Early batches may be incomplete or biased
>* Smoothing avoids zero probabilities and overconfidence

>* Smooth updates keep batch learning consistent
>* New features can enter gradually

>* Match smoothing to data quality and pace
>* Balance stability with responsiveness to new patterns



In [ ]:
#@title Python Code - Smoothing for Updates

# Incremental Naive Bayes needs cautious smoothing.
# Small batches can miss important words.
# This example compares two smoothing strengths.

import math
import numpy as np

# Define tiny count batches for two classes.
classes = ["ham", "spam"]
vocab = ["hello", "offer", "winner"]
class_index = {name: idx for idx, name in enumerate(classes)}

# Store batches as word counts and labels.
batches = [
    ([3, 0, 0], "ham"),
    ([0, 2, 0], "spam"),
    ([0, 0, 2], "spam"),
]

# Create a small prediction document.
test_counts = np.array([0, 0, 1], dtype=float)
assert test_counts.shape[0] == len(vocab)

# Update counts like partial_fit would do.
def update_counts(word_counts, class_counts, counts, label):
    row = class_index[label]
    word_counts[row] += np.array(counts, dtype=float)
    class_counts[row] += 1.0
    return word_counts, class_counts

# Predict with multinomial Naive Bayes smoothing.
def predict_spam_probability(word_counts, class_counts, alpha):
    smoothed = word_counts + alpha
    totals = smoothed.sum(axis=1, keepdims=True)
    feature_log_probs = np.log(smoothed / totals)

    priors = (class_counts + alpha) / (class_counts.sum() + alpha * 2)
    log_scores = np.log(priors) + feature_log_probs @ test_counts
    shifted = log_scores - log_scores.max()

    probabilities = np.exp(shifted) / np.exp(shifted).sum()
    return float(probabilities[class_index["spam"]])

# Run incremental updates for two alpha values.
for alpha in [0.01, 1.0]:
    word_counts = np.zeros((2, 3), dtype=float)
    class_counts = np.zeros(2, dtype=float)

    print(f"alpha={alpha}: smoothing strength")
    for step, (counts, label) in enumerate(batches, start=1):
        word_counts, class_counts = update_counts(word_counts, class_counts, counts, label)
        spam_prob = predict_spam_probability(word_counts, class_counts, alpha)
        print(f"  after batch {step}: P(spam | winner) = {spam_prob:.3f}")

print("Stronger smoothing changes early updates more cautiously.")



### **3.2. Priors and Weights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_02.jpg?v=1783967944" width="250">



>* Priors set initial class frequency expectations
>* They stabilize learning from uneven batches

>* Weights control each example’s update influence
>* Prioritize reliable, recent, or rare-class data

>* Coordinate priors and weights carefully
>* Monitor streams and adjust update influence



In [ ]:
#@title Python Code - Priors and Weights

# Incremental Naive Bayes can learn batch by batch.
# Priors set starting beliefs about class frequency.
# Weights change how strongly examples influence updates.

import numpy as np

# Show the NumPy version for reproducibility.
print("NumPy version:", np.__version__)

# Create tiny count features for two classes.
X_batches = [
    np.array([[3, 0], [2, 1], [0, 3]]),
    np.array([[1, 2], [0, 4], [4, 0]])
]

y_batches = [
    np.array([0, 0, 1]),
    np.array([1, 1, 0])
]

# Give trusted examples more influence during updates.
w_batches = [
    np.array([1.0, 1.0, 2.0]),
    np.array([1.0, 2.0, 1.0])
]

# Set class priors and smoothing strength.
class_prior = np.array([0.7, 0.3])
alpha = 1.0

# Initialize weighted class and feature counts.
class_count = class_prior.copy()
feature_count = np.zeros((2, 2), dtype=float)

# Update counts one mini batch at a time.
for X, y, weights in zip(X_batches, y_batches, w_batches):
    assert X.shape[0] == y.shape[0] == weights.shape[0]
    for row, label, weight in zip(X, y, weights):
        class_count[label] += weight
        feature_count[label] += row * weight

# Convert counts into Naive Bayes probabilities.
class_prob = class_count / class_count.sum()
feature_prob = (feature_count + alpha)
feature_prob /= feature_prob.sum(axis=1, keepdims=True)

# Predict one new count vector safely.
new_email = np.array([2, 1])
log_scores = np.log(class_prob)
log_scores += new_email @ np.log(feature_prob).T
prediction = int(np.argmax(log_scores))

# Print compact results for the lesson.
print("Weighted class probabilities:", np.round(class_prob, 3))
print("Smoothed feature probabilities:", np.round(feature_prob, 3))
print("Prediction for [2, 1] counts:", prediction)



### **3.3. Incremental partial fit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_A/image_03_03.jpg?v=1783967942" width="250">



>* Train Naive Bayes in manageable batches
>* Update models without full retraining

>* Batches gradually update Naive Bayes statistics
>* Declare all possible classes before training

>* Batch quality affects early model behavior
>* Monitor drift and update strategy regularly



In [ ]:
#@title Python Code - Incremental partial fit

# Demonstrate incremental Naive Bayes learning.
# Use tiny batches for clarity.
# Avoid external machine learning libraries.

import numpy as np

# Create small count based training data.
X_batches = [
    np.array([[3, 0, 1], [2, 1, 0]]),
    np.array([[0, 3, 1], [1, 2, 1]]),
    np.array([[4, 0, 0], [0, 4, 1]]),
]

y_batches = [
    np.array([0, 0]),
    np.array([1, 1]),
    np.array([0, 1]),
]

# Define all possible classes upfront.
classes = np.array([0, 1])
alpha = 1.0

# Initialize class and feature counts.
class_count = np.zeros(classes.size)
feature_count = np.zeros((classes.size, 3))

# Update counts one batch at a time.
for batch_id, (X, y) in enumerate(zip(X_batches, y_batches), start=1):
    assert X.shape[0] == y.shape[0]
    for class_index, label in enumerate(classes):
        mask = y == label
        class_count[class_index] += mask.sum()
        feature_count[class_index] += X[mask].sum(axis=0)
    print(f"After batch {batch_id}: class counts {class_count.astype(int).tolist()}")

# Convert accumulated counts into probabilities.
smoothed = feature_count + alpha
feature_prob = smoothed / smoothed.sum(axis=1, keepdims=True)
class_prior = class_count / class_count.sum()

# Predict one new count based message.
new_item = np.array([2, 0, 1])
log_scores = np.log(class_prior) + new_item @ np.log(feature_prob).T
predicted_class = classes[np.argmax(log_scores)]

# Show compact final model behavior.
print("Feature probabilities by class:", np.round(feature_prob, 2).tolist())
print("Predicted class for new item:", int(predicted_class))



# <font color="#418FDE" size="6.5" uppercase>**Naive Bayes**</font>


In this lecture, you learned to:
- Choose an appropriate Naive Bayes variant for numeric, count, binary, or categorical features. 
- Configure smoothing, priors, and sample weights in Naive Bayes models. 
- Use partial_fit for incremental Naive Bayes training. 

In the next Lecture (Lecture B), we will go over 'Probability Calibration'